Objective:
To classify images in the CIFAR10 dataset using convolutional neural networks.

Data:
CIFAR10 is a subset of the 80 million tiny images dataset. They were collected by Alex Krizhevsky, Vinod Nair, and Geoffrey Hinton. The dataset is available at CIFAR-10 and CIFAR-100 datasets (toronto.edu) and can also be loaded directly from TensorFlow using tf.keras.datasets.cifar10.load_data.

Problem Statement:
Image classification is an important part of computer vision systems. Equipped with a digital camera and a single board computer (such as a Raspberry Pi), smart technology can capture an image, determine what is in the image using a classification model, and then take an action based on that information. As a warmup exercise to develop such technology, consider that you are tasked to classify images from the CIFAR10 dataset. You are to build your own CNN model for this, both from scratch and from an existing model via transfer learning and fine tuning.


In [ ]:
import tensorflow as tf
import numpy as np
from tensorflow.keras import layers
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import tensorflow.keras as keras
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix

# Data
1. Load CIFAR10 dataset into training and testing, features and labels numpy arrays using cifar10.load_data. Using markdown, list the 10 classes.

In [ ]:

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()


In [ ]:
# Print off shape
print("Training features shape:", x_train.shape)
print("Training labels shape:", y_train.shape)
print("Test features shape:", x_test.shape)
print("Test labels shape:", y_test.shape)

CIFAR-10 Classes
|     | Class |
| -------- | ------- |
| 1 |  Airplane   |
| 2 |  Automobile    |
| 3 |  Bird   |
| 4 |  Cat   |
| 5 |  Deer    |
| 6 |  Dog   |
| 7 |  Frog   |
| 8 |  Horse    |
| 9 |  Ship   |
| 10 | Truck    |

2. Create a bar plot using seaborn.barplot of the number of elements in each category of the entire dataset. Use markdown to comment on how well balanced the dataset is.

In [ ]:
# Flatten and combine all labels
y_all = np.concatenate([y_train, y_test]).flatten()

# Class names
class_names = ['Airplane', 'Automobile', 'Bird', 'Cat', 'Deer',
               'Dog', 'Frog', 'Horse', 'Ship', 'Truck']

# Create countplot
plt.figure(figsize=(10, 6))
sns.countplot(x=y_all, palette='viridis')
plt.xticks(ticks=range(10), labels=class_names, rotation=45)
plt.title('CIFAR-10 Full Dataset Class Distribution')
plt.xlabel('Classes')
plt.ylabel('Number of Samples')
plt.tight_layout()
plt.show()

Dataset is perfectly balanced

3. Use sklearn.model_selection.train_test_split to split the test set into test and validation sets choosing appropriate proportions. 

In [ ]:
# split the test set into test and validation sets
x_val, x_test_new, y_val, y_test_new = train_test_split(
    x_test, y_test, 
    test_size=0.75, 
    random_state=42,
    stratify=y_test
)

print(f"Validation shape: {x_val.shape}")    
print(f"Test shape: {x_test_new.shape}")    

4. Create train, test, and validation data generators using tensorflow.keras.preprocessing.image.ImageDataGenerator; each should scale the data by dividing by 255, and the training generator should also use data augmentation. 

In [ ]:
# Rescale all by /255
train_datagen = ImageDataGenerator(
    rescale=1/255,
    rotation_range=15,      # Random rotation up to 15 degrees
    width_shift_range=0.1,  # Horizontal shift
    height_shift_range=0.1, # Vertical shift
    horizontal_flip=True,   # Random horizontal flips
    zoom_range=0.1          # Random zoom
)

val_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

# Batch that divides evenly
batch_size = 25

# Create generators using flow() for NumPy arrays
train_generator = train_datagen.flow(
    x_train, y_train,
    batch_size=batch_size,
    shuffle=True
)

val_generator = val_datagen.flow(
    x_val, y_val,
    batch_size=batch_size,
    shuffle=False
)

test_generator = test_datagen.flow(
    x_test_new, y_test_new,
    batch_size=batch_size,
    shuffle=False
)


# Modeling 
1. Use tf.keras.Sequential to create a convolutional neural network. Use at least two convolution layers and at least two pooling layers. Choose an activation function for each layer, and make sure the input and output dimensions are appropriate for the data. Print a summary of the model using tf.summary.

In [ ]:
# Instantiate model with two convolution layers and two pooling layers
model = tf.keras.Sequential([
    layers.Conv2D(32, 3, activation='relu', input_shape=(32, 32, 3)),
    layers.MaxPooling2D(),
    layers.Conv2D(32, 3, activation='relu'),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax')
])

model.summary()

2. Compile the model with a choice of optimizer, sparse_categorical_crossentropy for the loss function, and set the metrics argument equal to ['accuracy'].

In [ ]:
# compile model with adam optimizer, sparse_categorical_crossentropy and accuracy for the metric
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

3. Train the model using the train and validation data generators; record the training accuracy. Experiment with different architectures other hyperparameters to improve upon the results.

In [ ]:

# Train with exact steps (run after your generators)
steps_per_epoch = len(x_train) // batch_size
val_steps = len(x_val) // batch_size  

# Run the model with train and validation generator sets
history = model.fit(
    train_generator,
    steps_per_epoch=steps_per_epoch,
    epochs=10, 
    validation_data=val_generator,
    validation_steps=val_steps,
    verbose=1
)

# Capture accuracy and print results per epoch
train_acc = history.history['accuracy'][-1]
print(f"Baseline final training accuracy: {train_acc:.4f}")

### Hyperparameter Tuning

In [ ]:
# Instantiate model with two convolution layers and two pooling layers
model = tf.keras.Sequential([
    layers.Conv2D(32, 3, activation='relu', input_shape=(32, 32, 3)),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation='relu'),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax')
])

model.summary()

In [ ]:
# comile model with adam optimizer, sparse_categorical_crossentropy and accuracy for the metric
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:

# Train with exact steps (run after your generators)
steps_per_epoch = len(x_train) // batch_size 
val_steps = len(x_val) // batch_size

# Run the model with train and validation generator sets
history = model.fit(
    train_generator,
    steps_per_epoch=steps_per_epoch,
    epochs=10, 
    validation_data=val_generator,
    validation_steps=val_steps,
    verbose=1
)

# Capture accuracy and print results per epoch
train_acc = history.history['accuracy'][-1]
print(f"Baseline final training accuracy: {train_acc:.4f}")

Through extensive testing 0.726 was the lowest val_accuracy achievable

4. Start a new model by loading one of the models from tensorflow.keras.applications along with the pretrained weights; don't include the top layer. Check if your model comes with preprocess_input function and be sure to use that properly with your data before training. Describe the model you chose using markdown and explain why you think it will work well for this use case.

We chose MobileNetV2 from tensorflow.keras.applications, a lightweight pretrained model ideal for CIFAR-10 transfer learning without heavy computation.

MobileNetV2 is pretrained on ImageNet (1.4M images, 1000 classes) and excels on small RGB images like CIFAR-10's 32x32x3 via efficient depthwise separable convolutions.

| Feature    | Details                                                                                                                                                                       |
| ---------- | ----------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| Base       | MobileNetV2(weights='imagenet', include_top=False, input_shape=(32,32,3))                                                                                    |
| Params     | ~2.2M trainable (after top); total ~3.5M tensorflow​                                                                                                                          |
| Layers     | 155 conv/inverted residual blocks + GlobalAvgPool2D                                                                                                                           |
| Preprocess | tf.keras.applications.mobilenet_v2.preprocess_input scales/normalizes to ImageNet stats (mean subtraction) github​                                                            |
| Why Chosen | Efficient (fast on CPU/GPU), strong low-level features (edges/textures) transfer well to CIFAR-10 objects |

In [ ]:
import tensorflow.keras.applications as apps
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

x_train_resized = tf.image.resize(x_train, (96,96)).numpy()
x_val_resized = tf.image.resize(x_val, (96,96)).numpy()
x_test_resized = tf.image.resize(x_test_new, (96,96)).numpy()

# New generators with MobileNetV2 preprocess
train_datagen_mob = ImageDataGenerator(
    preprocessing_function=preprocess_input,  # ImageNet stats
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    zoom_range=0.1
)

# New generators for the uploaded model
val_datagen_mob = ImageDataGenerator(preprocessing_function=preprocess_input)
test_datagen_mob = ImageDataGenerator(preprocessing_function=preprocess_input)

train_generator_mob = train_datagen_mob.flow(x_train_resized, y_train, batch_size=batch_size, shuffle=True)
val_generator_mob = val_datagen_mob.flow(x_val_resized, y_val, batch_size=batch_size, shuffle=False)
test_generator_mob = test_datagen_mob.flow(x_test_resized, y_test_new, batch_size=batch_size, shuffle=False)


In [ ]:
# Instantiate MobileNetV2 and make it untrainable 
base_model = apps.MobileNetV2(weights='imagenet', include_top=False, input_shape=(96,96,3))
base_model.trainable = False


9412608/9406464 [==============================] - 0s 0us/step


5. Add on a new top layer with appropriate hyperparameter choices. Choose a number of layers to freeze. Print a summary of the model.

In [ ]:
# After the existing model_mob definition and before the second .compile()

# Enhanced top layers with BatchNorm and additional Dense for better performance
model_mob = tf.keras.Sequential([
    base_model,
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax')
])

# Choose to freeze first 80% of layers 
num_layers = len(base_model.layers)
freeze_until_layer = int(0.8 * num_layers)
for layer in base_model.layers[:freeze_until_layer]:
    layer.trainable = False

# Print model summary
model_mob.summary()


6. Compile the model with a choice of optimizer and loss function, and the set the metrics argument equal to ['accuracy'].

In [ ]:

# Re-compile with lower learning rate suitable for fine-tuning
model_mob.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.0001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print(f"Frozen first {freeze_until_layer} layers out of {num_layers} in base model.")



7. Train the model using the train and validation data generators; record the training accuracy. Experiment with different architectures, different numbers of frozen layers, and other hyperparameters to improve upon the results.

In [ ]:
print("Training enhanced model_mob...")
history_enhanced = model_mob.fit(
    train_generator_mob,
    steps_per_epoch=steps_per_epoch,
    epochs=10,  # More epochs to leverage improvements
    validation_data=val_generator_mob,
    validation_steps=val_steps,
    verbose=1
)

# Record final training accuracy
train_acc_enhanced = history_enhanced.history['accuracy'][-1]
val_acc_enhanced = history_enhanced.history['val_accuracy'][-1]
print(f"Final training accuracy: {train_acc_enhanced:.4f}")
print(f"Final validation accuracy: {val_acc_enhanced:.4f}")


### Hyperparameter Tuning

In [ ]:
# After the existing model_mob definition and before the second .compile()

# Enhanced top layers with BatchNorm and additional Dense for better performance
model_mob = tf.keras.Sequential([
    base_model,
    layers.Flatten(),
    layers.Dense(32, activation='relu'),
    layers.Dense(10, activation='softmax')
])

# Choose to freeze first 80% of layers 
num_layers = len(base_model.layers)
freeze_until_layer = int(0.5 * num_layers)
for layer in base_model.layers[:freeze_until_layer]:
    layer.trainable = False

# Print model summary
model_mob.summary()

In [ ]:

# Re-compile with lower learning rate suitable for fine-tuning
model_mob.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.0001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print(f"Frozen first {freeze_until_layer} layers out of {num_layers} in base model.")

In [ ]:
# Train the enhanced model_mob (with 80% frozen layers, improved top) using the mobile generators
steps_per_epoch = len(x_train) // batch_size  
val_steps = len(x_val) // batch_size          

print("Training enhanced model_mob...")
history_enhanced = model_mob.fit(
    train_generator_mob,
    steps_per_epoch=steps_per_epoch,
    epochs=10,
    validation_data=val_generator_mob,
    validation_steps=val_steps,
    verbose=1
)

# Record final training accuracy
train_acc_enhanced = history_enhanced.history['accuracy'][-1]
val_acc_enhanced = history_enhanced.history['val_accuracy'][-1]
print(f"Final training accuracy: {train_acc_enhanced:.4f}")
print(f"Final validation accuracy: {val_acc_enhanced:.4f}")


# Conclusion
1. Find the training and validation accuracies, and validation confusion matrix for both the custom CNN and transfer learning models. Present the results for both neatly. Use markdown to compare them and select the best model.

In [ ]:
# CIFAR-10 class names
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

# For Custom CNN
print("Custom CNN Results:")
custom_train_acc = history.history['accuracy'][-1]
custom_val_acc = history.history['val_accuracy'][-1]
print(f"Training Accuracy: {custom_train_acc:.4f}")
print(f"Validation Accuracy: {custom_val_acc:.4f}")

val_generator.reset()
y_val_pred_custom = model.predict(val_generator, steps=val_steps, verbose=0)
y_val_pred_classes_custom = np.argmax(y_val_pred_custom, axis=1)

#  confusion matrix
cm_custom = confusion_matrix(y_val[:len(y_val_pred_classes_custom)], y_val_pred_classes_custom)
print("\nCustom CNN Validation Confusion Matrix:")
print(cm_custom)
print("Row sums (true class counts):", cm_custom.sum(axis=1))

# For Transfer Learning MobileNetV2
print("\nTransfer Learning Results:")
mob_train_acc = history_enhanced.history['accuracy'][-1]
mob_val_acc = history_enhanced.history['val_accuracy'][-1]
print(f"Training Accuracy: {mob_train_acc:.4f}")
print(f"Validation Accuracy: {mob_val_acc:.4f}")

val_generator_mob.reset()
y_val_pred_mob = model_mob.predict(val_generator_mob, steps=val_steps, verbose=0)
y_val_pred_classes_mob = np.argmax(y_val_pred_mob, axis=1)

# confusion matrix
cm_mob = confusion_matrix(y_val[:len(y_val_pred_classes_mob)], y_val_pred_classes_mob)
print("\nMobileNetV2 Validation Confusion Matrix:")
print(cm_mob)
print("Row sums (true class counts):", cm_mob.sum(axis=1))

# Plot heatmaps
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.heatmap(cm_custom, annot=True, fmt='d', ax=axes[0], cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
axes[0].set_title('Custom CNN Val CM')
sns.heatmap(cm_mob, annot=True, fmt='d', ax=axes[1], cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
axes[1].set_title('MobileNetV2 Val CM')
plt.tight_layout()
plt.show()

2. Find the testing accuracy and confusion matrix of only the best model.

In [ ]:
best_model = model

test_generator.reset()
y_test_pred = model.predict(test_generator, verbose=0)
y_test_pred_classes = np.argmax(y_test_pred, axis=1)

test_acc = accuracy_score(y_test_new, y_test_pred_classes)
print(f"Test Accuracy: {test_acc:.4f}")

cm_test = confusion_matrix(y_test_new, y_test_pred_classes)
print("\n Best CNN Testing Confusion Matrix:")
print(cm_test)
print("Row sums (true class counts):", cm_test.sum(axis=1))

3. Use the model to make predictions on at least three other images from one of the 10 classes.

In [ ]:
import random
# Choosing three random images from the testing set
rand_images = random.sample(range(len(x_test)), 3)
# Displaying them
plt.figure(figsize=(10,3))

# Loops through the three images and find their actual label from the testing set
for i, index_test in enumerate(rand_images):
    img = x_test[index_test]
    true_label = class_names[y_test[index_test][0]]
# if len(y_test.shape)>1 else y_test[idx]

# Resizing and expanding the image
    img_resized = img / 255.0
    img_input = np.expand_dims(img_resized, axis=0)

# Making predictions and find the label
    probability = model.predict(img_input, verbose=0)
    prediction = class_names[np.argmax(probability)]

# Shows the images, prediction and true classification
    plt.subplot(1,3,i+1)
    plt.imshow(img)
    plt.title(f"True label: {true_label}, \n Predicted label: {prediction}")
    plt.axis("off")

plt.tight_layout()
plt.show()

4. Use markdown to comment on how well the model works to make predictions for this use case.

The custom CNN model achieved a training accuracy of 0.6776 and a validation accuracy of 0.7176 after hyperparameter tuning. This shows that it is not overfitting. Compared to the transfer learning model, it is performing heaps better since the latter could only achieve 0.3567 and 0.0888 training and validation accuracies, exhibiting intense overfitting and just poor performance in general. Therefore, the custom model was selected as the best performing one.

This model achieved a test accuracy of 71.04%, exhibiting good generalization. It seems to mess up mostly on pictures of birds, cats and deers from the confusion matrix, while does considerably well on classifying the objects. This means it has a slightly more difficult time classifying animals than inanimate objects (which might be because those animals tend to not be very visually distinct). It has also performed well on the random images, classifying all of them correctly.